In [ ]:
## Stage 0 — Setup

Imports and header so worth athletics doesn't pick it up as a bot.

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from io import StringIO

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

In [ ]:
## Stage 1 — Scrape a single race
Here we scrap the races from the results page and get this information:
athlete, country, time, record flag 
- reaction time
- wind reading
- date, venue, meet name

Handles non-numeric results (DNF/DQ/DNS) by keeping the row with a NaN time,
since a DNF is still a real data point (something went wrong that race).

In [ ]:
def clean_mark(mark):
    """Split '9.79 PB' into (9.79, 'PB'), or '9.91' into (9.91, None).
    Non-numeric marks (DNF/DQ/DNS) get NaN time and the code as the flag."""
    if pd.isna(mark):
        return np.nan, None
    parts = str(mark).strip().split()
    try:
        time = float(parts[0])
        flag = parts[1] if len(parts) > 1 else None
    except ValueError:
        time = np.nan
        flag = parts[0]  # e.g. 'DNF', 'DQ', 'DNS'
    return time, flag


def scrape_race(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()

    tables = pd.read_html(StringIO(resp.text))
    results_table = max(tables, key=len)

    results_table[["time", "record_flag"]] = results_table["MARK"].apply(
        lambda m: pd.Series(clean_mark(m))
    )

    soup = BeautifulSoup(resp.text, "html.parser")

    # wind
    heading = soup.find(string=lambda s: s and "Wind" in s)
    wind = None
    if heading:
        match = re.search(r"Wind\s*([+-]?\d+\.\d+)", heading)
        if match:
            wind = float(match.group(1))

    # date — from EventDate meta tag
    date = None
    for meta in soup.find_all("meta"):
        if meta.get("name") == "EventDate" or meta.get("property") == "EventDate":
            date = meta.get("content")
            break

    # venue + meet name
    desc_tag = soup.find("meta", attrs={"property": "og:description"})
    venue = None
    if desc_tag and desc_tag.get("content"):
        venue_match = re.search(r"in\s+([A-Za-z\s]+)$", desc_tag["content"].strip())
        if venue_match:
            venue = venue_match.group(1).strip()

    title_tag = soup.find("meta", attrs={"property": "og:title"})
    meet_name = title_tag["content"].strip() if title_tag and title_tag.get("content") else None

    results_table["wind"] = wind
    results_table["date"] = date
    results_table["venue"] = venue
    results_table["meet_name"] = meet_name
    results_table["source_url"] = url

    return results_table[["ATHLETE", "COUNTRY", "time", "record_flag", "Reaction Time",
                            "wind", "date", "venue", "meet_name", "source_url"]]

Found 1 tables on the page


,POS,BIB,ATHLETE,COUNTRY,MARK,ms*,Reaction Time
0,1,3007,Noah LYLES,USA,9.83 =WL,NaN,0.145
1,2,2102,Letsile TEBOGO,BOT,9.88 NR,0.873,0.129
2,3,2350,Zharnel HUGHES,GBR,9.88,0.874,0.105
3,4,2554,Oblique SEVILLE,JAM,9.88,0.877,0.148
4,5,2978,Christian COLEMAN,USA,9.92,NaN,0.137
5,6,2587,Abdul Hakim SANI BROWN,JPN,10.04,NaN,0.177
6,7,2626,Ferdinand OMANYALA,KEN,10.07,NaN,0.152
7,8,2540,Ryiem FORDE,JAM,10.08,NaN,0.188


In [32]:
import re  # for pulling the wind number out of a sentence

soup = BeautifulSoup(resp.text, "html.parser")

heading = soup.find(string=lambda s: s and "Wind" in s) # search all the text on the page for anything containing "Wind"
print(heading) 



                        Wind 0.0
                


In [33]:
# now grab just the number after "Wind"
wind = None
if heading:
    match = re.search(r"Wind\s*(-?\d+\.\d+)", heading)
    if match:
        wind = float(match.group(1))

print("wind:", wind)

wind: 0.0


In [34]:
import numpy as np

def clean_mark(mark):
    """Split '9.79 PB' into (9.79, 'PB'), or '9.91' into (9.91, None)"""
    if pd.isna(mark):
        return np.nan, None
    parts = str(mark).strip().split()
    time = float(parts[0])
    flag = parts[1] if len(parts) > 1 else None
    return time, flag

results_table[["time", "record_flag"]] = results_table["MARK"].apply(
    lambda m: pd.Series(clean_mark(m))
)

results_table[["ATHLETE", "COUNTRY", "time", "record_flag", "Reaction Time"]]

,ATHLETE,COUNTRY,time,record_flag,Reaction Time
0,Noah LYLES,USA,9.83,=WL,0.145
1,Letsile TEBOGO,BOT,9.88,NR,0.129
2,Zharnel HUGHES,GBR,9.88,NaN,0.105
3,Oblique SEVILLE,JAM,9.88,NaN,0.148
4,Christian COLEMAN,USA,9.92,NaN,0.137
5,Abdul Hakim SANI BROWN,JPN,10.04,NaN,0.177
6,Ferdinand OMANYALA,KEN,10.07,NaN,0.152
7,Ryiem FORDE,JAM,10.08,NaN,0.188


In [35]:
def scrape_race(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    
    tables = pd.read_html(StringIO(resp.text))
    results_table = max(tables, key=len)
    
    results_table[["time", "record_flag"]] = results_table["MARK"].apply(
        lambda m: pd.Series(clean_mark(m))
    )
    
    soup = BeautifulSoup(resp.text, "html.parser")
    
    # wind
    heading = soup.find(string=lambda s: s and "Wind" in s)
    wind = None
    if heading:
        match = re.search(r"Wind\s*([+-]?\d+\.\d+)", heading)
        if match:
            wind = float(match.group(1))
    
    # date — from EventDate meta tag, not page text
    date = None
    for meta in soup.find_all("meta"):
        if meta.get("name") == "EventDate" or meta.get("property") == "EventDate":
            date = meta.get("content")
            break
    
    # venue + meet name
    desc_tag = soup.find("meta", attrs={"property": "og:description"})
    venue = None
    if desc_tag and desc_tag.get("content"):
        venue_match = re.search(r"in\s+([A-Za-z\s]+)$", desc_tag["content"].strip())
        if venue_match:
            venue = venue_match.group(1).strip()
    
    title_tag = soup.find("meta", attrs={"property": "og:title"})
    meet_name = title_tag["content"].strip() if title_tag and title_tag.get("content") else None
    
    results_table["wind"] = wind
    results_table["date"] = date
    results_table["venue"] = venue
    results_table["meet_name"] = meet_name
    results_table["source_url"] = url
    
    return results_table[["ATHLETE", "COUNTRY", "time", "record_flag", "Reaction Time",
                            "wind", "date", "venue", "meet_name", "source_url"]]

In [36]:
race1 = scrape_race("https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result")
race2 = scrape_race("https://worldathletics.org/results/olympic-games/2020/the-xxxii-olympic-games-7153115/men/100-metres/final/result")
race3 = scrape_race("https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result")

pd.concat([race1, race2, race3])

,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
0,Noah LYLES,USA,9.79,PB,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
2,Fred KERLEY,USA,9.81,SB,0.108,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
3,Akani SIMBINE,RSA,9.82,NR,0.149,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
4,Lamont Marcell JACOBS,ITA,9.85,SB,0.114,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
5,Letsile TEBOGO,BOT,9.86,NR,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
6,Kenneth BEDNAREK,USA,9.88,NaN,0.163,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
7,Oblique SEVILLE,JAM,9.91,NaN,0.171,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
0,Noah LYLES,USA,9.79,PB,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...


In [37]:
# Search the whole page text for anything date-shaped
import re

page_text = soup.get_text()
date_candidates = re.findall(r"\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4}", page_text)
print(date_candidates)

['27 Aug 2023']


In [38]:
resp = requests.get(
    "https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result",
    headers=HEADERS, timeout=15
)
soup_paris = BeautifulSoup(resp.text, "html.parser")
page_text = soup_paris.get_text()

all_dates = re.findall(r"\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4}", page_text)
print(all_dates)

['27 Aug 2023']


In [39]:
idx = page_text.find("27 Aug 2023")
print(page_text[idx-150:idx+150])

liers

















 Countdown
                    





                    World Athletics Championships
                    Budapest23
19 Aug - 27 Aug 2023



















Introduction
Structure
Employment
Tenders
Contact
Newsletters

























Introduction
Next Courses
Endura


In [40]:
import json

scripts = soup_paris.find_all("script", type="application/ld+json")
print(f"Found {len(scripts)} JSON-LD blocks")
for s in scripts:
    print(s.string[:500] if s.string else None)
    print("---")

Found 1 JSON-LD blocks

        {
        "@context": "http://schema.org",
        "@type": "WebSite",
        "url": "https://worldathletics.org/",
        "potentialAction": {
        "@type": "SearchAction",
        "target": "https://worldathletics.org/search/?q={search_term_string}",
        "query-input": "required name=search_term_string"
        }
        }
    
---


In [41]:
time_tags = soup_paris.find_all("time")
for t in time_tags:
    print(t.get("datetime"), "|", t.get_text(strip=True))

In [42]:
for meta in soup_paris.find_all("meta"):
    if meta.get("property", "").startswith("article") or "date" in meta.get("property", "").lower() or "date" in meta.get("name", "").lower():
        print(meta.get("property") or meta.get("name"), ":", meta.get("content"))

EventDate : 01/08/2024 00:00:00


In [43]:
race_paris = scrape_race("https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result")
race_budapest = scrape_race("https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result")

pd.concat([race_paris, race_budapest])

,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
0,Noah LYLES,USA,9.79,PB,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
2,Fred KERLEY,USA,9.81,SB,0.108,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
3,Akani SIMBINE,RSA,9.82,NR,0.149,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
4,Lamont Marcell JACOBS,ITA,9.85,SB,0.114,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
5,Letsile TEBOGO,BOT,9.86,NR,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
6,Kenneth BEDNAREK,USA,9.88,NaN,0.163,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
7,Oblique SEVILLE,JAM,9.91,NaN,0.171,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
0,Noah LYLES,USA,9.83,=WL,0.145,0.0,19/08/2023 00:00:00,Budapest,World Athletics | 100 Metres Result | World A...,https://worldathletics.org/results/world-athle...
1,Letsile TEBOGO,BOT,9.88,NR,0.129,0.0,19/08/2023 00:00:00,Budapest,World Athletics | 100 Metres Result | World A...,https://worldathletics.org/results/world-athle...


In [44]:
urls = [
    "https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result",  # Paris 2024
    "https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result",  # Budapest 2023
    "https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result",  # Tokyo 2020(21)
    "https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result",  # Oregon 2022 - different URL structure, good test
]

all_races = []
for u in urls:
    try:
        race = scrape_race(u)
        all_races.append(race)
        print(f"OK ({len(race)} rows): {u}")
    except Exception as e:
        print(f"FAILED: {u}")
        print(f"   error: {e}")

full_dataset = pd.concat(all_races, ignore_index=True)
full_dataset

OK (8 rows): https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result
FAILED: https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result
   error: could not convert string to float: 'DNF'
FAILED: https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result
   error: 'MARK'


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
0,Noah LYLES,USA,9.79,PB,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
2,Fred KERLEY,USA,9.81,SB,0.108,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
3,Akani SIMBINE,RSA,9.82,NR,0.149,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
4,Lamont Marcell JACOBS,ITA,9.85,SB,0.114,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
5,Letsile TEBOGO,BOT,9.86,NR,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
6,Kenneth BEDNAREK,USA,9.88,NaN,0.163,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
7,Oblique SEVILLE,JAM,9.91,NaN,0.171,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
8,Noah LYLES,USA,9.83,=WL,0.145,0.0,19/08/2023 00:00:00,Budapest,World Athletics | 100 Metres Result | World A...,https://worldathletics.org/results/world-athle...
9,Letsile TEBOGO,BOT,9.88,NR,0.129,0.0,19/08/2023 00:00:00,Budapest,World Athletics | 100 Metres Result | World A...,https://worldathletics.org/results/world-athle...
